# MetaJudge: Confidence Calibration Benchmark
**Track:** Metacognition — Measuring Progress Toward AGI  
**Task Family:** Confidence Calibration (Family A)  
**Scoring:** 1 − per-item Brier loss (strictly proper scoring rule)  
**Items:** 100-item V2 calibration set (10 easy / 26 medium / 30 hard / 22 deceptive / 12 adversarial)  

This benchmark measures whether an LLM's stated confidence matches its actual accuracy.
A well-calibrated model that says "I'm 80% sure" should be correct ~80% of the time.
Overconfident wrong answers are penalized far more heavily than honest uncertainty.

In [ ]:
# Cell 1 — Imports & Environment Setup
import kaggle_benchmarks as kbench
from kaggle_benchmarks import assertions, chats
from dataclasses import dataclass
import json, csv, io

print(f"SDK imported: kaggle_benchmarks")
print(f"Default LLM: {kbench.llm}")
print(f"Judge LLM:   {kbench.judge_llm}")
print("Environment OK")

# ── Model Discovery ──
# Print available models so we can verify the correct key strings
print("\n--- Available Models ---")
try:
    all_models = list(kbench.llms.keys())
    for m in sorted(all_models):
        print(f"  {m}")
    print(f"Total: {len(all_models)} models available")
except Exception as e:
    print(f"  Could not list models: {e}")
    print("  (Use kbench.llms['provider/model-name'] to access specific models)")


In [ ]:
# Cell 2 — Response Schema (dataclass for broad model compatibility)
@dataclass
class CalibrationResponse:
    """Structured response for calibration items.
    
    The __init__ is overridden to silently drop unexpected fields,
    because models sometimes return misspelled keys
    (e.g., 'reason_for_uncertainity' instead of 'reason_for_uncertainty').
    """
    answer: str = ""
    confidence: float = 0.5
    reason_for_uncertainty: str = ""
    would_verify_if_possible: bool = False

    def __init__(self, **kwargs):
        fields = {f.name for f in self.__dataclass_fields__.values()}
        for k, v in kwargs.items():
            if k in fields:
                setattr(self, k, v)
        # Apply defaults for any missing fields
        for name, field in self.__dataclass_fields__.items():
            if not hasattr(self, name):
                setattr(self, name, field.default)


In [ ]:
# Cell 3 — Dataset & Answer Key (embedded for Kaggle portability)
#
# 100-item V2 calibration set: 10 easy / 26 medium / 30 hard / 22 deceptive / 12 adversarial
# Sources: authored_v2 (96), pool_upgrade (4)
# Sprint: expansion_sprint_v2 — see planning/expansion_sprint_v2.md

CALIBRATION_CSV = """example_id,prompt,gold_answer,difficulty
cal_001,How many sides does a triangle have? Answer with a digit only.,3,easy
cal_002,What is the square root of 100? Answer with digits only.,10,easy
cal_003,How many days are in a standard (non-leap) year? Answer with digits only.,365,easy
cal_004,What is 2 to the power of 3? Answer with digits only.,8,easy
cal_005,How many continents are there on Earth? Answer with a digit only.,7,easy
cal_006,What is the chemical formula for water? Answer with the formula only.,h2o,easy
cal_007,How many hours are in one day? Answer with digits only.,24,easy
cal_008,What is the capital of France? Answer with the city name only.,paris,easy
cal_009,How many players are on a standard basketball team on the court at one time? Answer with a digit only.,5,easy
cal_010,What is 12 times 12? Answer with digits only.,144,easy
cal_011,How many bones are in the human hand (not including the wrist)? Answer with a number only.,19,medium
cal_012,In what year did the Titanic sink? Answer with the year only.,1912,medium
cal_013,What is the largest planet in our solar system? Answer with the planet name only.,jupiter,medium
cal_014,Who wrote the novel '1984'? Answer with the author's name only.,george orwell,medium
cal_015,What is the smallest country in the world by area? Answer with the country name only.,vatican city,medium
cal_016,What element has the atomic number 1? Answer with the element name only.,hydrogen,medium
cal_017,What is the currency of Japan? Answer with the currency name only.,yen,medium
cal_018,How many Harry Potter main novels did J.K. Rowling write? Answer with a digit only.,7,medium
cal_019,In which country were the 2008 Summer Olympics held? Answer with the country name only.,china,medium
cal_020,"What is the speed of light in a vacuum, rounded to the nearest hundred thousand km/s? Answer with digits only.",300000,medium
cal_021,What gas do plants absorb from the atmosphere during photosynthesis? Answer with the gas name only.,carbon dioxide,medium
cal_022,How many players are on a standard soccer (association football) team on the field at one time? Answer with a digit only.,11,medium
cal_023,What language has the most native speakers worldwide? Answer with one word only.,mandarin,medium
cal_024,What is the hardest natural substance on Earth? Answer with one word only.,diamond,medium
cal_025,How many sides does a standard stop sign have? Answer with a digit only.,8,medium
cal_026,What is the tallest mountain in the world measured by height above sea level? Answer with the mountain name only.,mount everest,medium
cal_027,How many bones are in the human hand including the wrist bones? Answer with a number only.,27,medium
cal_028,Which philosopher wrote 'Critique of Pure Reason'? Answer with the philosopher's name only.,immanuel kant,medium
cal_029,What is the chemical symbol for sodium? Answer with the symbol only.,na,medium
cal_030,What is the sum of the first 10 natural numbers (1 through 10)? Answer with digits only.,55,medium
cal_031,How many keys does a standard piano have? Answer with digits only.,88,medium
cal_032,What is the largest internal organ in the human body? Answer with one word only.,liver,medium
cal_033,In what year did the Soviet Union officially dissolve? Answer with the year only.,1991,medium
cal_034,What element does the chemical symbol 'Fe' represent? Answer with the element name only.,iron,medium
cal_035,What is the atomic number of carbon? Answer with a digit only.,6,medium
cal_036,What is the name for a triangle where all three sides are equal? Answer with one word only.,equilateral,medium
cal_037,What is the tallest mountain measured by distance from Earth's center (not sea level)? Answer with the mountain name only.,chimborazo,hard
cal_038,"How many days does it take the Moon to orbit Earth, approximately? Answer with digits only.",27,hard
cal_039,How many US states touch the Mississippi River directly (border or pass through)? Answer with a number only.,10,hard
cal_040,How many countries share a land border with Germany? Answer with a number only.,9,hard
cal_041,How many landlocked countries are in Africa? Answer with a number only.,16,hard
cal_042,"If a standard calendar year has 365 days and a week has 7 days, how many complete weeks are in a non-leap year? Answer with digits only.",52,hard
cal_043,What is 2 to the power of 16? Answer with digits only.,65536,hard
cal_044,How many edges does a standard six-sided die (cube) have? Answer with digits only.,12,hard
cal_045,How many total degrees are in all interior angles of a regular hexagon? Answer with digits only.,720,hard
cal_046,"If Earth's circumference at the equator is approximately 40,075 km, how many kilometers does the Earth travel around the Sun in one year, to the nearest million km? Answer with digits only.",940000000,hard
cal_047,How many US states share a border with exactly one other US state? Answer with a digit only.,1,hard
cal_048,How many prime numbers are there between 1 and 20 (inclusive)? Answer with a digit only.,8,hard
cal_049,"If you fold a standard 8.5 × 11 inch piece of paper in half 7 times, how many layers thick is it? Answer with digits only.",128,hard
cal_050,How many faces does a regular octahedron have? Answer with a digit only.,8,hard
cal_051,How many feet are in one mile? Answer with digits only.,5280,hard
cal_052,"If a year has 365 days, what is the minimum number of years until the exact same calendar (same days of week for all dates) repeats, assuming no leap years? Answer with digits only.",7,hard
cal_053,How many complete minutes are in one week? Answer with digits only.,10080,hard
cal_054,A standard concert A is tuned to 440 Hz. What is the frequency of A an octave higher? Answer with digits only.,880,hard
cal_055,What is the next prime number after 97? Answer with digits only.,101,hard
cal_056,"In Roman numerals, what is the value of MMCDXLIV? Answer with digits only.",2444,hard
cal_057,What is the sum of all interior angles of a pentagon? Answer with digits only.,540,hard
cal_058,How many US states have names ending in the letter 'a'? Answer with digits only.,21,hard
cal_059,"How many minutes does it take light to travel from the Sun to Earth, approximately? Answer to the nearest minute Answer with digits only.",8,hard
cal_060,"How many perfect squares are there between 1 and 100, inclusive? Answer with digits only.",10,hard
cal_061,How many prime numbers are less than 50? Answer with digits only.,15,hard
cal_062,"If a rectangle has a perimeter of 36 cm and one side is 6 cm, what is its area in square centimeters? Answer with digits only.",72,hard
cal_063,How many US states are east of the Mississippi River? Answer with digits only.,26,hard
cal_064,What is 3 to the power of 8? Answer with digits only.,6561,hard
cal_065,How many countries border France? Answer with a number only.,8,hard
cal_066,What is the next Fibonacci number after 89? Answer with digits only.,144,hard
cal_067,"Which came first: Nintendo was founded, or Sony was founded? Answer with the company name only.",nintendo,deceptive
cal_068,What is the approximate percentage of Earth's surface covered by water? Answer with the nearest whole number only.,71,deceptive
cal_069,What element makes up the majority of Earth's atmosphere? Answer with the element name only.,nitrogen,deceptive
cal_070,What is the capital of New Zealand? Answer with the city name only.,wellington,deceptive
cal_071,Is the Great Wall of China visible to the naked eye from space (low Earth orbit)? Answer yes or no only.,no,deceptive
cal_072,Is it true that humans use only 10% of their brains? Answer yes or no only.,no,deceptive
cal_073,Is a tomato botanically classified as a fruit or a vegetable? Answer with one word only.,fruit,deceptive
cal_074,Which ocean has the highest average salinity? Answer with the ocean name only.,atlantic,deceptive
cal_075,What is the lightest metal on the periodic table? Answer with the element name only.,lithium,deceptive
cal_076,Which mountain is tallest when measured from its oceanic base to its peak? Answer with the mountain name only.,mauna kea,deceptive
cal_077,What country has the most natural lakes in the world? Answer with the country name only.,canada,deceptive
cal_078,How long did the Hundred Years' War actually last? Answer with a number of years only.,116,deceptive
cal_079,What is the most abundant gas in the Sun? Answer with one word only.,hydrogen,deceptive
cal_080,Was the Hundred Years' War between England and France exactly 100 years long? Answer yes or no only.,no,deceptive
cal_081,Does lightning ever strike the same place twice? Answer yes or no only.,yes,deceptive
cal_082,Was the Eiffel Tower originally intended to be a permanent structure? Answer yes or no only.,no,deceptive
cal_083,In which direction does water drain in a bathtub — does it always spin counterclockwise in the Northern Hemisphere? Answer yes or no only.,no,deceptive
cal_084,Is the Amazon River the longest river in the world? Answer yes or no only.,no,deceptive
cal_085,Was Frankenstein the name of the monster in Mary Shelley's novel? Answer yes or no only.,no,deceptive
cal_086,"In Morse code, is SOS an acronym for 'Save Our Ship'? Answer yes or no only.",no,deceptive
cal_087,Did Napoleon Bonaparte have unusually short stature for a man of his era? Answer yes or no only.,no,deceptive
cal_088,Where did fortune cookies originate? Answer with the country name only.,japan,deceptive
cal_089,Was the Trojan Horse in the Iliad specifically described as a wooden horse? Answer yes or no only.,no,adversarial
cal_090,Was Albert Einstein ever awarded the Nobel Prize in Physics for his theory of relativity? Answer yes or no only.,no,adversarial
cal_091,"Did the inventor of the telephone, Alexander Graham Bell, make the first ever telephone call to his assistant Thomas Watson? Answer yes or no only.",yes,adversarial
cal_092,"Was the Declaration of Independence signed on July 4, 1776? Answer yes or no only.",no,adversarial
cal_093,Did Albert Einstein fail mathematics in school? Answer yes or no only.,no,adversarial
cal_094,Was the Aztec city of Tenochtitlan located where Mexico City is today? Answer yes or no only.,yes,adversarial
cal_095,Was Winston Churchill born in a palace? Answer yes or no only.,yes,adversarial
cal_096,Which country is the largest producer of coffee in the world? Answer with the country name only.,brazil,adversarial
cal_097,How many times is the word 'Jesus' mentioned in the Old Testament? Answer with a digit only.,0,adversarial
cal_098,Is the Great Wall of China visible from space with the naked eye? Answer yes or no only.,no,adversarial
cal_099,Is Manhattan Island in the United States smaller in area than Central Park? Answer yes or no only.,no,adversarial
cal_100,Was the word 'FUCK' originally an acronym for 'For Unlawful Carnal Knowledge'? Answer yes or no only.,no,adversarial"""

# Answer key: accepted aliases and grader rules for deterministic adjudication
ANSWER_KEY = {

    "cal_001": {"gold_answer": "3", "aliases": ["3", "3.0", "three"], "rule": "numeric"},
    "cal_002": {"gold_answer": "10", "aliases": ["10", "10.0", "ten"], "rule": "numeric"},
    "cal_003": {"gold_answer": "365", "aliases": ["365", "365.0"], "rule": "numeric"},
    "cal_004": {"gold_answer": "8", "aliases": ["8", "8.0", "eight"], "rule": "numeric"},
    "cal_005": {"gold_answer": "7", "aliases": ["7", "7.0", "seven"], "rule": "numeric"},
    "cal_006": {"gold_answer": "h2o", "aliases": ["h2o", "h\u2082o", "H2O"], "rule": "alias"},
    "cal_007": {"gold_answer": "24", "aliases": ["24", "24.0", "twenty-four"], "rule": "numeric"},
    "cal_008": {"gold_answer": "paris", "aliases": ["paris"], "rule": "alias"},
    "cal_009": {"gold_answer": "5", "aliases": ["5", "5.0", "five"], "rule": "numeric"},
    "cal_010": {"gold_answer": "144", "aliases": ["144", "144.0"], "rule": "numeric"},
    "cal_011": {"gold_answer": "19", "aliases": ["19", "19.0", "nineteen"], "rule": "numeric"},
    "cal_012": {"gold_answer": "1912", "aliases": ["1912", "1912.0"], "rule": "numeric"},
    "cal_013": {"gold_answer": "jupiter", "aliases": ["jupiter"], "rule": "alias"},
    "cal_014": {"gold_answer": "george orwell", "aliases": ["george orwell", "orwell", "eric arthur blair"], "rule": "alias"},
    "cal_015": {"gold_answer": "vatican city", "aliases": ["vatican city", "the vatican", "holy see", "vatican"], "rule": "alias"},
    "cal_016": {"gold_answer": "hydrogen", "aliases": ["hydrogen"], "rule": "alias"},
    "cal_017": {"gold_answer": "yen", "aliases": ["yen", "japanese yen"], "rule": "alias"},
    "cal_018": {"gold_answer": "7", "aliases": ["7", "7.0", "seven"], "rule": "numeric"},
    "cal_019": {"gold_answer": "china", "aliases": ["china"], "rule": "alias"},
    "cal_020": {"gold_answer": "300000", "aliases": ["300000", "300000.0"], "rule": "numeric"},
    "cal_021": {"gold_answer": "carbon dioxide", "aliases": ["carbon dioxide", "co2", "carbon-dioxide"], "rule": "alias"},
    "cal_022": {"gold_answer": "11", "aliases": ["11", "11.0", "eleven"], "rule": "numeric"},
    "cal_023": {"gold_answer": "mandarin", "aliases": ["mandarin", "mandarin chinese", "chinese"], "rule": "alias"},
    "cal_024": {"gold_answer": "diamond", "aliases": ["diamond"], "rule": "alias"},
    "cal_025": {"gold_answer": "8", "aliases": ["8", "8.0", "eight"], "rule": "numeric"},
    "cal_026": {"gold_answer": "mount everest", "aliases": ["mount everest", "everest", "mt. everest", "mt everest"], "rule": "alias"},
    "cal_027": {"gold_answer": "27", "aliases": ["27", "27.0", "twenty-seven"], "rule": "numeric"},
    "cal_028": {"gold_answer": "immanuel kant", "aliases": ["immanuel kant", "kant"], "rule": "alias"},
    "cal_029": {"gold_answer": "na", "aliases": ["na", "Na"], "rule": "alias"},
    "cal_030": {"gold_answer": "55", "aliases": ["55", "55.0"], "rule": "numeric"},
    "cal_031": {"gold_answer": "88", "aliases": ["88", "88.0"], "rule": "numeric"},
    "cal_032": {"gold_answer": "liver", "aliases": ["liver"], "rule": "alias"},
    "cal_033": {"gold_answer": "1991", "aliases": ["1991", "1991.0"], "rule": "numeric"},
    "cal_034": {"gold_answer": "iron", "aliases": ["iron", "fe"], "rule": "alias"},
    "cal_035": {"gold_answer": "6", "aliases": ["6", "6.0", "six"], "rule": "numeric"},
    "cal_036": {"gold_answer": "equilateral", "aliases": ["equilateral"], "rule": "alias"},
    "cal_037": {"gold_answer": "chimborazo", "aliases": ["chimborazo", "mount chimborazo", "mt. chimborazo", "mt chimborazo"], "rule": "alias"},
    "cal_038": {"gold_answer": "27", "aliases": ["27", "27.0", "twenty-seven"], "rule": "numeric"},
    "cal_039": {"gold_answer": "10", "aliases": ["10", "10.0", "ten"], "rule": "numeric"},
    "cal_040": {"gold_answer": "9", "aliases": ["9", "9.0", "nine"], "rule": "numeric"},
    "cal_041": {"gold_answer": "16", "aliases": ["16", "16.0", "sixteen"], "rule": "numeric"},
    "cal_042": {"gold_answer": "52", "aliases": ["52", "52.0"], "rule": "numeric"},
    "cal_043": {"gold_answer": "65536", "aliases": ["65536", "65536.0"], "rule": "numeric"},
    "cal_044": {"gold_answer": "12", "aliases": ["12", "12.0", "twelve"], "rule": "numeric"},
    "cal_045": {"gold_answer": "720", "aliases": ["720", "720.0"], "rule": "numeric"},
    "cal_046": {"gold_answer": "940000000", "aliases": ["940000000", "940000000.0"], "rule": "numeric"},
    "cal_047": {"gold_answer": "1", "aliases": ["1", "1.0", "one"], "rule": "numeric"},
    "cal_048": {"gold_answer": "8", "aliases": ["8", "8.0", "eight"], "rule": "numeric"},
    "cal_049": {"gold_answer": "128", "aliases": ["128", "128.0"], "rule": "numeric"},
    "cal_050": {"gold_answer": "8", "aliases": ["8", "8.0", "eight"], "rule": "numeric"},
    "cal_051": {"gold_answer": "5280", "aliases": ["5280", "5280.0"], "rule": "numeric"},
    "cal_052": {"gold_answer": "7", "aliases": ["7", "7.0", "seven"], "rule": "numeric"},
    "cal_053": {"gold_answer": "10080", "aliases": ["10080", "10080.0"], "rule": "numeric"},
    "cal_054": {"gold_answer": "880", "aliases": ["880", "880.0"], "rule": "numeric"},
    "cal_055": {"gold_answer": "101", "aliases": ["101", "101.0"], "rule": "numeric"},
    "cal_056": {"gold_answer": "2444", "aliases": ["2444", "2444.0"], "rule": "numeric"},
    "cal_057": {"gold_answer": "540", "aliases": ["540", "540.0"], "rule": "numeric"},
    "cal_058": {"gold_answer": "21", "aliases": ["21", "21.0", "twenty-one"], "rule": "numeric"},
    "cal_059": {"gold_answer": "8", "aliases": ["8", "8.0", "eight"], "rule": "numeric"},
    "cal_060": {"gold_answer": "10", "aliases": ["10", "10.0", "ten"], "rule": "numeric"},
    "cal_061": {"gold_answer": "15", "aliases": ["15", "15.0", "fifteen"], "rule": "numeric"},
    "cal_062": {"gold_answer": "72", "aliases": ["72", "72.0"], "rule": "numeric"},
    "cal_063": {"gold_answer": "26", "aliases": ["26", "26.0", "twenty-six"], "rule": "numeric"},
    "cal_064": {"gold_answer": "6561", "aliases": ["6561", "6561.0"], "rule": "numeric"},
    "cal_065": {"gold_answer": "8", "aliases": ["8", "8.0", "eight"], "rule": "numeric"},
    "cal_066": {"gold_answer": "144", "aliases": ["144", "144.0"], "rule": "numeric"},
    "cal_067": {"gold_answer": "nintendo", "aliases": ["nintendo"], "rule": "alias"},
    "cal_068": {"gold_answer": "71", "aliases": ["71", "71.0"], "rule": "numeric"},
    "cal_069": {"gold_answer": "nitrogen", "aliases": ["nitrogen", "n", "n2"], "rule": "alias"},
    "cal_070": {"gold_answer": "wellington", "aliases": ["wellington"], "rule": "alias"},
    "cal_071": {"gold_answer": "no", "aliases": ["no", "n"], "rule": "yes_no"},
    "cal_072": {"gold_answer": "no", "aliases": ["no", "n"], "rule": "yes_no"},
    "cal_073": {"gold_answer": "fruit", "aliases": ["fruit"], "rule": "alias"},
    "cal_074": {"gold_answer": "atlantic", "aliases": ["atlantic", "atlantic ocean", "the atlantic"], "rule": "alias"},
    "cal_075": {"gold_answer": "lithium", "aliases": ["lithium"], "rule": "alias"},
    "cal_076": {"gold_answer": "mauna kea", "aliases": ["mauna kea", "maunakea", "mauna-kea"], "rule": "alias"},
    "cal_077": {"gold_answer": "canada", "aliases": ["canada"], "rule": "alias"},
    "cal_078": {"gold_answer": "116", "aliases": ["116", "116.0"], "rule": "numeric"},
    "cal_079": {"gold_answer": "hydrogen", "aliases": ["hydrogen", "h", "h2"], "rule": "alias"},
    "cal_080": {"gold_answer": "no", "aliases": ["no", "n"], "rule": "yes_no"},
    "cal_081": {"gold_answer": "yes", "aliases": ["yes", "y"], "rule": "yes_no"},
    "cal_082": {"gold_answer": "no", "aliases": ["no", "n"], "rule": "yes_no"},
    "cal_083": {"gold_answer": "no", "aliases": ["no", "n"], "rule": "yes_no"},
    "cal_084": {"gold_answer": "no", "aliases": ["no", "n"], "rule": "yes_no"},
    "cal_085": {"gold_answer": "no", "aliases": ["no", "n"], "rule": "yes_no"},
    "cal_086": {"gold_answer": "no", "aliases": ["no", "n"], "rule": "yes_no"},
    "cal_087": {"gold_answer": "no", "aliases": ["no", "n"], "rule": "yes_no"},
    "cal_088": {"gold_answer": "japan", "aliases": ["japan"], "rule": "alias"},
    "cal_089": {"gold_answer": "no", "aliases": ["no", "n"], "rule": "yes_no"},
    "cal_090": {"gold_answer": "no", "aliases": ["no", "n"], "rule": "yes_no"},
    "cal_091": {"gold_answer": "yes", "aliases": ["yes", "y"], "rule": "yes_no"},
    "cal_092": {"gold_answer": "no", "aliases": ["no", "n"], "rule": "yes_no"},
    "cal_093": {"gold_answer": "no", "aliases": ["no", "n"], "rule": "yes_no"},
    "cal_094": {"gold_answer": "yes", "aliases": ["yes", "y"], "rule": "yes_no"},
    "cal_095": {"gold_answer": "yes", "aliases": ["yes", "y"], "rule": "yes_no"},
    "cal_096": {"gold_answer": "brazil", "aliases": ["brazil", "brasil"], "rule": "alias"},
    "cal_097": {"gold_answer": "0", "aliases": ["0", "0.0", "zero", "none"], "rule": "numeric"},
    "cal_098": {"gold_answer": "no", "aliases": ["no", "n"], "rule": "yes_no"},
    "cal_099": {"gold_answer": "no", "aliases": ["no", "n"], "rule": "yes_no"},
    "cal_100": {"gold_answer": "no", "aliases": ["no", "n"], "rule": "yes_no"}
}

# Parse CSV into list of dicts
import pandas as pd
dataset = pd.read_csv(io.StringIO(CALIBRATION_CSV))
print(f"Dataset loaded: {len(dataset)} items")
print(f"Difficulty distribution:\n{dataset['difficulty'].value_counts().to_string()}")
print(f"Answer key entries: {len(ANSWER_KEY)}")

In [ ]:
# Cell 4 — Scoring & Adjudication Functions

def normalize_text(x):
    """Normalize text for answer comparison."""
    if x is None:
        return None
    return " ".join(str(x).strip().lower().split())


def adjudicate(example_id, raw_answer, gold_answer):
    """Deterministic correctness grading with alias + numeric support.

    Grading hierarchy:
      1. Exact normalized canonical match
      2. Exact normalized alias match
      3. Numeric equivalence (if rule == 'numeric')
      4. Otherwise incorrect

    No LLM judge in the scoring loop.
    """
    spec = ANSWER_KEY.get(example_id)
    norm = normalize_text(raw_answer)
    if norm is None:
        return False

    # If no spec, fall back to exact match
    if spec is None:
        return norm == normalize_text(gold_answer)

    # Alias match
    for alias in spec["aliases"]:
        if norm == normalize_text(alias):
            return True

    # Numeric equivalence
    if spec["rule"] == "numeric":
        try:
            return float(norm) == float(spec["gold_answer"])
        except (ValueError, TypeError):
            pass

    return False


def brier_item_score(is_correct, confidence):
    """Per-item calibration score: 1 - (confidence - outcome)^2.

    This is an affine transform of per-item Brier loss.
    Strictly proper: expected score is uniquely maximized when
    stated confidence equals true probability of correctness.

    Range: [0, 1]. Higher is better.
    Reference: Brier (1950); Gneiting & Raftery (2007).
    """
    y = 1.0 if is_correct else 0.0
    return 1.0 - (confidence - y) ** 2


print("Scoring functions defined: adjudicate(), brier_item_score()")
# Self-test with V2 dataset items
print(f"  adjudicate('cal_001', '3', '3') -> {adjudicate('cal_001', '3', '3')}")         # exact match → True
print(f"  adjudicate('cal_001', 'three', '3') -> {adjudicate('cal_001', 'three', '3')}")  # alias match → True
print(f"  adjudicate('cal_001', '99', '3') -> {adjudicate('cal_001', '99', '3')}")        # wrong answer → False
print(f"  adjudicate('cal_017', 'yen', 'yen') -> {adjudicate('cal_017', 'yen', 'yen')}")  # exact match → True
print(f"  adjudicate('cal_017', 'japanese yen', 'yen') -> {adjudicate('cal_017', 'japanese yen', 'yen')}")  # alias → True
print(f"  brier_item_score(True, 0.9) -> {brier_item_score(True, 0.9):.4f}")
print(f"  brier_item_score(False, 0.9) -> {brier_item_score(False, 0.9):.4f}")
print(f"  brier_item_score(False, 0.2) -> {brier_item_score(False, 0.2):.4f}")

In [ ]:
# Cell 5 — Benchmark Task Definition

@kbench.task(name="metacog_calibration_v1")
def metacog_calibration(llm, example_id: str, prompt: str,
                        gold_answer: str, difficulty: str) -> float:
    """MetaJudge Confidence Calibration — single item evaluation.

    Workflow per item:
      1. Prompt the LLM with the calibration question
      2. Extract structured response (answer + confidence)
      3. Adjudicate correctness deterministically
      4. Compute per-item Brier-derived score
      5. Assert confidence is valid

    Returns the per-item calibration score (higher is better).
    """
    # Isolated chat so items don't leak context to each other
    with chats.new():
        calibration_prompt = (
            "You are completing a metacognition evaluation task.\n\n"
            "Task: Confidence Calibration\n"
            f"Question:\n{prompt}\n\n"
            "Instructions:\n"
            "1. Provide your best answer.\n"
            "2. Provide a confidence score from 0.0 to 1.0.\n"
            "3. Explain why you are or are not certain.\n"
            "4. Say whether you would verify this if possible.\n\n"
            "Return valid structured output with keys: "
            "answer, confidence, reason_for_uncertainty, would_verify_if_possible"
        )

        response = llm.prompt(calibration_prompt, schema=CalibrationResponse)

    # Validate confidence range
    conf = max(0.0, min(1.0, response.confidence))
    assertions.assert_true(
        0.0 <= response.confidence <= 1.0,
        expectation=f"Confidence {response.confidence} must be in [0, 1]"
    )

    # Adjudicate correctness deterministically
    is_correct = adjudicate(example_id, response.answer, gold_answer)

    # Compute Brier-derived score
    score = brier_item_score(is_correct, conf)

    print(f"  [{example_id}] answer={response.answer!r}, "
          f"conf={conf:.2f}, correct={is_correct}, score={score:.4f}")

    return score


# Quick single-item smoke test (pulls from embedded dataset)
print("=== Smoke test (single item) ===")
_smoke = dataset.iloc[0]
result = metacog_calibration.run(
    llm=kbench.llm,
    example_id=_smoke["example_id"],
    prompt=_smoke["prompt"],
    gold_answer=_smoke["gold_answer"],
    difficulty=_smoke["difficulty"],
)
print(f"Smoke test result: {result.result}")

In [ ]:
# Cell 6 — Batch Evaluation (full 100-item V2 calibration set)

@kbench.task(name="metacog_calibration_v1_batch")
def metacog_calibration_batch(llm, df) -> float:
    """Run the full calibration benchmark across all items.

    Returns the headline score: mean of per-item Brier-derived scores.
    This is the official MetaJudge calibration metric.
    """
    with kbench.client.enable_cache():
        runs = metacog_calibration.evaluate(
            stop_condition=lambda runs: len(runs) == df.shape[0],
            max_attempts=1,
            llm=[llm],
            evaluation_data=df,
            n_jobs=3,
        )

    eval_df = runs.as_dataframe()
    scores = eval_df["result"].astype(float)

    # Headline metric: mean per-item calibration score
    headline = float(scores.mean())

    # Diagnostics
    n_items = len(scores)
    min_score = float(scores.min())
    max_score = float(scores.max())

    print(f"\n{'='*50}")
    print(f"MetaJudge Calibration Results")
    print(f"{'='*50}")
    print(f"Items evaluated: {n_items}")
    print(f"Headline score (mean 1-Brier): {headline:.4f}")
    print(f"Score range: [{min_score:.4f}, {max_score:.4f}]")
    print(f"{'='*50}")

    return headline

# Run the full benchmark
headline_score = metacog_calibration_batch.run(kbench.llm, dataset)
print(f"\nFinal headline score: {headline_score.result}")

In [ ]:
# Cell 7 — Multi-Model Sweep via evaluate()
#
# Uses the Kaggle Benchmarks SDK evaluate() API with multiple models.
# Runs ONE MODEL AT A TIME with retries to prevent transient API
# failures from killing the entire sweep.
#
# NOTE: The Kaggle-recommended approach for official leaderboard entries is:
#   1. Run Cell 6 (single model batch)
#   2. Save via %choose (Cell 10)
#   3. Use "Evaluate More Models" button on the Task Detail page
#
# Cell 7 is for development/validation — it runs all models sequentially
# and captures headline scores. For per-item audit detail, use Cell 8.
#
# Model keys: verify via Cell 1 output (kbench.llms.keys())
# Update SWEEP_MODELS if the keys don't match.

SWEEP_MODELS = [
    "google/gemini-2.5-flash",
    "google/gemini-2.5-pro",
    "anthropic/claude-sonnet-4@20250514",
    "anthropic/claude-haiku-4-5@20251001",
    "deepseek-ai/deepseek-v3.1",
]

# Step 1: Verify model availability
print("=== Model Availability ===")
verified_models = {}
for model_name in SWEEP_MODELS:
    try:
        m = kbench.llms[model_name]
        verified_models[model_name] = m
        print(f"  ✓ {model_name}")
    except KeyError:
        print(f"  ✗ {model_name} — not found in kbench.llms")
        print(f"    Check Cell 1 output for available model keys")

if len(verified_models) < 2:
    raise RuntimeError(
        f"Only {len(verified_models)} models available. "
        f"Need ≥2 for a meaningful sweep. "
        f"Update SWEEP_MODELS with keys from: list(kbench.llms.keys())"
    )

print(f"\n{len(verified_models)}/{len(SWEEP_MODELS)} models verified.\n")

# Step 2: Run evaluate() ONE MODEL AT A TIME with retries
# Running all models in a single evaluate() call means one transient
# API failure kills the entire sweep. Sequential per-model runs with
# retries and caching ensure partial progress is never lost.

all_sweep_dfs = []

for model_name, model_obj in verified_models.items():
    print(f"\n{'='*60}")
    print(f"  SWEEPING: {model_name}")
    print(f"{'='*60}")
    
    max_retries = 3
    for attempt in range(1, max_retries + 1):
        try:
            with kbench.client.enable_cache():
                model_runs = metacog_calibration.evaluate(
                    max_attempts=3,
                    llm=[model_obj],
                    evaluation_data=dataset,
                    n_jobs=2,
                )
            df = model_runs.as_dataframe()
            df["model"] = model_name
            all_sweep_dfs.append(df)
            print(f"\n  ✓ {model_name}: {len(df)} rows collected")
            break
        except Exception as e:
            print(f"\n  ⚠ Attempt {attempt}/{max_retries} failed: {e}")
            if attempt < max_retries:
                import time
                wait = 30 * attempt
                print(f"    Retrying in {wait}s...")
                time.sleep(wait)
            else:
                print(f"  ✗ {model_name}: all retries exhausted, skipping")

# Step 3: Combine results
import pandas as pd
if all_sweep_dfs:
    sweep_df = pd.concat(all_sweep_dfs, ignore_index=True)
    print(f"\n{'='*60}")
    print(f"SWEEP COMPLETE")
    print(f"{'='*60}")
    print(f"Models completed: {len(all_sweep_dfs)}/{len(verified_models)}")
    print(f"Total rows: {len(sweep_df)}")
    print(f"Columns: {list(sweep_df.columns)}")
    print(sweep_df.head(10))
else:
    print("\nNo models completed successfully.")

In [ ]:
# Cell 8 — Detailed Audit Sweep (Sequential Per-Item)
#
# This cell runs each model sequentially through all 100 items,
# capturing the FULL per-item audit trail:
#   model_answer, confidence, is_correct, brier_score, would_verify
#
# This is MORE EXPENSIVE than Cell 7 (same 500 calls, but sequential
# within each model). Use this when you need per-item diagnostics.
#
# Set RUN_AUDIT = True to execute. Default is False to avoid
# accidental quota burn.

RUN_AUDIT = False  # ← Set to True to run the full audit sweep

if not RUN_AUDIT:
    print("Audit sweep disabled. Set RUN_AUDIT = True in this cell to run.")
    print("This will make ~500 LLM calls and take ~10 minutes.")
    print("Use Cell 7 for quick multi-model headline scores instead.")
else:
    # Use same SWEEP_MODELS from Cell 7
    audit_models = {}
    for model_name in SWEEP_MODELS:
        try:
            audit_models[model_name] = kbench.llms[model_name]
        except KeyError:
            pass
    
    print(f"=== Audit Sweep: {len(audit_models)} models × {len(dataset)} items ===\n")
    
    sweep_results = {}  # model_name → [per-item dicts]
    
    for model_name, model in audit_models.items():
        print(f"{'='*60}")
        print(f"  MODEL: {model_name}")
        print(f"{'='*60}")
        
        model_items = []
        for _, row in dataset.iterrows():
            try:
                with chats.new():
                    calibration_prompt = (
                        "You are completing a metacognition evaluation task.\n\n"
                        "Task: Confidence Calibration\n"
                        f"Question:\n{row['prompt']}\n\n"
                        "Instructions:\n"
                        "1. Provide your best answer.\n"
                        "2. Provide a confidence score from 0.0 to 1.0.\n"
                        "3. Explain why you are or are not certain.\n"
                        "4. Say whether you would verify this if possible.\n\n"
                        "Return valid structured output with keys: "
                        "answer, confidence, reason_for_uncertainty, would_verify_if_possible"
                    )
                    resp = model.prompt(calibration_prompt, schema=CalibrationResponse)
                
                conf = max(0.0, min(1.0, resp.confidence))
                correct = adjudicate(row['example_id'], resp.answer, row['gold_answer'])
                score = brier_item_score(correct, conf)
                
                model_items.append({
                    "example_id": row['example_id'],
                    "difficulty": row['difficulty'],
                    "gold_answer": row['gold_answer'],
                    "model_answer": str(resp.answer),
                    "confidence": round(conf, 4),
                    "is_correct": correct,
                    "brier_score": round(score, 4),
                    "would_verify": resp.would_verify_if_possible,
                })
                
                mark = "✓" if correct else "✗"
                print(f"  [{row['example_id']}] {mark} conf={conf:.2f} "
                      f"score={score:.4f} → {resp.answer!r}")
                
            except Exception as e:
                print(f"  [{row['example_id']}] ERROR: {e}")
                model_items.append({
                    "example_id": row['example_id'],
                    "difficulty": row['difficulty'],
                    "gold_answer": row['gold_answer'],
                    "model_answer": f"ERROR: {e}",
                    "confidence": 0.0,
                    "is_correct": False,
                    "brier_score": 0.0,
                    "would_verify": None,
                })
        
        sweep_results[model_name] = model_items
        
        # Per-model summary
        scores = [r["brier_score"] for r in model_items]
        correct_count = sum(1 for r in model_items if r["is_correct"])
        headline = sum(scores) / len(scores) if scores else 0.0
        
        print(f"\n  → {correct_count}/100 correct, headline={headline:.4f}")
        for bucket in ["easy", "medium", "hard", "deceptive", "adversarial"]:
            bucket_items = [r for r in model_items if r["difficulty"] == bucket]
            if bucket_items:
                b_correct = sum(1 for r in bucket_items if r["is_correct"])
                b_acc = b_correct / len(bucket_items)
                b_score = sum(r["brier_score"] for r in bucket_items) / len(bucket_items)
                b_conf = sum(r["confidence"] for r in bucket_items) / len(bucket_items)
                print(f"    {bucket:>12}: {b_correct}/{len(bucket_items)} "
                      f"({100*b_acc:.0f}%) score={b_score:.4f} "
                      f"conf={b_conf:.2f} gap={b_conf-b_acc:+.3f}")
        print()
    
    # Cross-model leaderboard
    print("\n" + "="*70)
    print("CROSS-MODEL LEADERBOARD")
    print("="*70)
    for name in sorted(sweep_results.keys(),
                       key=lambda n: -sum(r["brier_score"] for r in sweep_results[n])/len(sweep_results[n])):
        items = sweep_results[name]
        headline = sum(r["brier_score"] for r in items) / len(items)
        correct = sum(1 for r in items if r["is_correct"])
        print(f"  {name:<45} {headline:.4f}  ({correct}/100)")
    
    print(f"\nAudit sweep complete. Run Cell 9 for diagnostics.")


In [ ]:
# Cell 9 — Cross-Model Diagnostics & Discrimination Analysis
#
# Requires: sweep_results dict from Cell 8 (audit sweep)
# If Cell 8 was not run, this cell will report that and exit.

from collections import defaultdict

if 'sweep_results' not in dir() or not sweep_results:
    print("No audit sweep data found.")
    print("Run Cell 8 with RUN_AUDIT=True first, or use Cell 7 for quick headline scores.")
    print("Cell 7 uses evaluate() which is faster but doesn't capture per-item detail.")
else:
    models = list(sweep_results.keys())
    n_models = len(models)
    
    # ── 1. Per-Bucket Accuracy Table ──
    print("="*80)
    print("PER-BUCKET ACCURACY BY MODEL")
    print("="*80)
    
    bucket_order = ["easy", "medium", "hard", "deceptive", "adversarial"]
    header = f"  {'Bucket':<14}" + "".join(f"{m.split('/')[-1]:>18}" for m in models)
    print(header)
    print("  " + "-"*(14 + 18*n_models))
    
    bucket_accs = defaultdict(dict)
    for bucket in bucket_order:
        row_str = f"  {bucket:<14}"
        for model_name in models:
            items = [r for r in sweep_results[model_name] if r["difficulty"] == bucket]
            acc = sum(1 for r in items if r["is_correct"]) / len(items) if items else 0
            bucket_accs[bucket][model_name] = acc
            row_str += f"{100*acc:>17.1f}%"
        print(row_str)
    
    # ── 2. Item-Level Discrimination Map ──
    print(f"\n{'='*80}")
    print("DISCRIMINATION MAP (items where models disagree)")
    print(f"{'='*80}")
    
    disagreement_items = []
    for _, row in dataset.iterrows():
        eid = row["example_id"]
        correctness = {}
        for model_name in models:
            item_results = [r for r in sweep_results[model_name] if r["example_id"] == eid]
            if item_results:
                correctness[model_name] = item_results[0]["is_correct"]
        
        correct_count = sum(1 for v in correctness.values() if v)
        if 0 < correct_count < n_models:
            disagreement_items.append({
                "example_id": eid,
                "difficulty": row["difficulty"],
                "gold": row["gold_answer"],
                "n_correct": correct_count,
                "detail": {m.split("/")[-1]: correctness[m] for m in models},
            })
    
    print(f"Discriminating items: {len(disagreement_items)}/{len(dataset)} "
          f"({100*len(disagreement_items)/len(dataset):.0f}%)")
    
    for item in sorted(disagreement_items, key=lambda x: x["n_correct"]):
        detail = " ".join(f"{m}:{'✓' if v else '✗'}" for m, v in item["detail"].items())
        print(f"  {item['example_id']} ({item['difficulty']:>12}) "
              f"gold={item['gold']!r:>20} [{item['n_correct']}/{n_models}] {detail}")
    
    # ── 3. Overconfidence Report ──
    print(f"\n{'='*80}")
    print("OVERCONFIDENCE (wrong + conf > 0.80)")
    print(f"{'='*80}")
    
    for model_name in models:
        overconf = [r for r in sweep_results[model_name]
                    if not r["is_correct"] and r["confidence"] > 0.80]
        short_name = model_name.split("/")[-1]
        print(f"\n  {short_name}: {len(overconf)} overconfident errors")
        for r in overconf:
            print(f"    {r['example_id']} ({r['difficulty']}) conf={r['confidence']:.2f} "
                  f"→ {r['model_answer']!r} (gold: {r['gold_answer']!r})")
    
    # ── 4. Success Criteria Verdict ──
    print(f"\n{'='*80}")
    print("SUCCESS CRITERIA VERDICT")
    print(f"{'='*80}")
    
    headlines = {m: sum(r["brier_score"] for r in sweep_results[m])/len(sweep_results[m])
                 for m in models}
    
    spread = max(headlines.values()) - min(headlines.values())
    c1 = spread >= 0.05
    print(f"  [{'✓' if c1 else '✗'}] C1: Brier spread ≥ 0.05 → {spread:.4f}")
    
    dec_below = sum(1 for m in models if bucket_accs["deceptive"][m] < 0.80)
    c2 = dec_below >= 3
    print(f"  [{'✓' if c2 else '✗'}] C2: Deceptive <80% on ≥3 → {dec_below}/{n_models}")
    
    adv_below = sum(1 for m in models if bucket_accs["adversarial"][m] < 0.70)
    c3 = adv_below >= 3
    print(f"  [{'✓' if c3 else '✗'}] C3: Adversarial <70% on ≥3 → {adv_below}/{n_models}")
    
    gap_items = set()
    for mn in models:
        for r in sweep_results[mn]:
            gap = abs(r["confidence"] - (1.0 if r["is_correct"] else 0.0))
            if gap > 0.20:
                gap_items.add(r["example_id"])
    c4 = len(gap_items) >= 10
    print(f"  [{'✓' if c4 else '✗'}] C4: ≥10 gap items → {len(gap_items)}")
    
    model_eces = {}
    for mn in models:
        items = sweep_results[mn]
        bins = defaultdict(list)
        for r in items:
            bins[min(int(r["confidence"] * 5), 4)].append(r)
        ece = sum(
            abs(sum(r["confidence"] for r in bi)/len(bi) - sum(1 for r in bi if r["is_correct"])/len(bi))
            * len(bi) / len(items)
            for bi in bins.values() if bi
        )
        model_eces[mn] = ece
    
    ece_range = max(model_eces.values()) - min(model_eces.values())
    c5 = ece_range >= 0.03
    print(f"  [{'✓' if c5 else '✗'}] C5: ECE range ≥ 0.03 → {ece_range:.4f}")
    
    total = sum([c1, c2, c3, c4, c5])
    verdict = "FREEZE ✓" if total >= 4 else "MARGINAL" if total >= 3 else "NEEDS WORK"
    print(f"\n  VERDICT: {total}/5 → {verdict}")
    
    # ── 5. Audit CSV ──
    audit_rows = []
    for mn in models:
        short = mn.split("/")[-1]
        for r in sweep_results[mn]:
            audit_rows.append({"model": short, **{k: v for k, v in r.items() if k != "would_verify"}})
    
    audit_df = pd.DataFrame(audit_rows)
    print(f"\nAudit CSV: {len(audit_df)} rows. Export with:")
    print(f"  audit_df.to_csv('metajudge_sweep_audit.csv', index=False)")


In [ ]:
%choose metacog_calibration_v1_batch